RFM Analysis


Import Libraries

In [2]:
import pandas as pd
import numpy as np 
import datetime as dt


Load Cleaned data


In [12]:
try:
    df = pd.read_excel("../data/cleaned_dataset.xlsx")
    print(df.head)
except Exception as e:
    print(e)

<bound method NDFrame.head of        Invoice StockCode                          Description  Quantity  \
0       489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1       489434    79323P                   PINK CHERRY LIGHTS        12   
2       489434    79323W                  WHITE CHERRY LIGHTS        12   
3       489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4       489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   
...        ...       ...                                  ...       ...   
525456  538171     22271                 FELTCRAFT DOLL ROSIE         2   
525457  538171     22750         FELTCRAFT PRINCESS LOLA DOLL         1   
525458  538171     22751       FELTCRAFT PRINCESS OLIVIA DOLL         1   
525459  538171     20970   PINK FLORAL FELTCRAFT SHOULDER BAG         2   
525460  538171     21931               JUMBO STORAGE BAG SUKI         2   

               InvoiceDate  Price  Customer ID         Country  
0   

In [15]:
df["TotalAmount"] = df["Quantity"] * df["Price"]
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
snapshot_date = df["InvoiceDate"].max() + dt.timedelta(days = 1)


In [17]:
rfm = df.groupby("Customer ID").agg({
    "InvoiceDate": lambda x: (snapshot_date-x.max()).days,
    "Invoice": "nunique",
    "TotalAmount": "sum"
}).reset_index()

rfm.columns = ['CustomerID','Recency','Frequency','Monetary']
rfm.head()

,CustomerID,Recency,Frequency,Monetary
0,12346.0,67,15,-64.68
1,12347.0,3,2,1323.32
2,12348.0,74,1,222.16
3,12349.0,43,4,2646.99
4,12351.0,11,1,300.93


In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# 1. Scaling (K-Means ke liye Recency, Frequency, Monetary ko scale karna zaroori hai)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# 2. K-Means Clustering (4 Segments banayenge)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# 3. Churn Target Variable Banayein (Agra Recency > 90 days hai toh Churned = 1, warna 0)
rfm['Churn'] = (rfm['Recency'] > 90).astype(int)

# Check results
print("--- Cluster-wise Customer Count ---")
print(rfm['Cluster'].value_counts())

print("\n--- Churn Count (1 = Churned, 0 = Active) ---")
print(rfm['Churn'].value_counts())

rfm.head()

--- Cluster-wise Customer Count ---
Cluster
0    3248
2    1076
3      54
1       5
Name: count, dtype: int64

--- Churn Count (1 = Churned, 0 = Active) ---
Churn
0    2925
1    1458
Name: count, dtype: int64


,CustomerID,Recency,Frequency,Monetary,Cluster,Churn
0,12346.0,67,15,-64.68,0,0
1,12347.0,3,2,1323.32,0,0
2,12348.0,74,1,222.16,0,0
3,12349.0,43,4,2646.99,0,0
4,12351.0,11,1,300.93,0,0
